In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
customer_data = [(1,'manish','patna',"30-05-2022"),
(2,'vikash','kolkata',"12-03-2023"),
(3,'nikita','delhi',"25-06-2023"),
(4,'rahul','ranchi',"24-03-2023"),
(5,'mahesh','jaipur',"22-03-2023"),
(6,'prantosh','kolkata',"18-10-2022"),
(7,'raman','patna',"30-12-2022"),
(8,'prakash','ranchi',"24-02-2023"),
(9,'ragini','kolkata',"03-03-2023"),
(10,'raushan','jaipur',"05-02-2023")]

customer_schema=['customer_id','customer_name','address','date_of_joining']

cust_df = spark.createDataFrame(customer_data,customer_schema)

In [0]:
sales_data = [(1,22,10,"01-06-2022"),
(1,27,5,"03-02-2023"),
(2,5,3,"01-06-2023"),
(5,22,1,"22-03-2023"),
(7,22,4,"03-02-2023"),
(9,5,6,"03-03-2023"),
(2,1,12,"15-06-2023"),
(1,56,2,"25-06-2023"),
(5,12,5,"15-04-2023"),
(11,12,76,"12-03-2023")]

sales_schema=['customer_id','product_id','quantity','date_of_purchase']

sales_df = spark.createDataFrame(sales_data, sales_schema)

In [0]:
product_data = [(1, 'fanta',20),
(2, 'dew',22),
(5, 'sprite',40),
(7, 'redbull',100),
(12,'mazza',45),
(22,'coke',27),
(25,'limca',21),
(27,'pepsi',14),
(56,'sting',10)]

product_schema=['id','name','price']

prod_df = spark.createDataFrame(product_data, product_schema)

In [0]:
cust_df.createOrReplaceTempView("customer")
sales_df.createOrReplaceTempView("sales")
prod_df.createOrReplaceTempView("product")

In [0]:
cust_df.show()

In [0]:
sales_df.show()

In [0]:
prod_df.show()

In [0]:
# customers that came but didnt bought any product 
# spark.sql(""" select distinct(customer_id)  from customer where customer_id not in (select distinct (customer_id) from sales)""").show()


# pyspark alternate using joins 
# cust_df.join(sales_df , on ='customer_id', how ='full')\
#     .dropDuplicates(['customer_id'])\
#     .filter("quantity is NULL")\
#     .select("customer_id")\
#     .show()

# alternate for pyspark :
cust_df.join(sales_df , cust_df['customer_id']==sales_df['customer_id'] , "inner")\
    .select(sales_df['customer_id'])\
    .show()

In [0]:
cust_df.join(sales_df, on = 'customer_id', how = 'left_anti').show()

In [0]:
# cust_df.join(sales_df, on = 'customer_id', how='outer').show()
cust_df.join(sales_df, cust_df['customer_id']==sales_df['customer_id'] ,'outer').show()

In [0]:
cust_df.join(sales_df, on = 'customer_id', how = 'left_semi').show()


In [0]:
cust_df.crossJoin(sales_df).count()


# Spark Join and Shuffle

Suppose we have two DataFrames:

* `DF1` = 500 MB → 4 partitions
* `DF2` = 500 MB → 4 partitions
* We have 2 worker nodes / executors
* We are joining both DataFrames on `id`

### 1. The problem

Initially, the data is distributed across different partitions and executors.

For example:

```text
Worker 1
 ├── DF1 Partition 1 → id = 1, 5, 8...
 ├── DF1 Partition 2 → id = 2, 7, 10...
 ├── DF2 Partition 1 → id = 3, 4, 6...
 └── DF2 Partition 2 → id = 1, 9, 11...

Worker 2
 ├── DF1 Partition 3 → id = 12, 15...
 ├── DF1 Partition 4 → id = 20, 25...
 ├── DF2 Partition 3 → id = 7, 14...
 └── DF2 Partition 4 → id = 30, 35...
```

Suppose `id = 1` is in:

```text
DF1 → Worker 1
DF2 → Worker 1
```

Then there is no problem because both records are already on the same worker.

But suppose:

```text
DF1 → Worker 1 → id = 7
DF2 → Worker 2 → id = 7
```

Now Spark needs to bring the matching records together.

---

# 2. Shuffle

This is where **shuffle** happens.

Spark redistributes the data based on the join key (`id`).

Conceptually, Spark does something like:

```text
hash(id) % number_of_shuffle_partitions
```

For example:

```text
id = 7 → Shuffle Partition 143
id = 7 → Shuffle Partition 143
```

Because both records have the same `id`, they will go to the same shuffle partition.

So:

```text
Before Shuffle:

Worker 1              Worker 2
   id=7                   id=7
    ↓                       ↓
    └─────────── SHUFFLE ───┘
                  ↓
          Same Shuffle Partition
                  ↓
               JOIN
```

---

# 3. Why is it expensive?

Shuffle becomes more expensive when data has to move **between different worker nodes**.

For example:

```text
Worker 1
    ↓
Network
    ↓
Worker 2
```

The data has to travel over the network from one machine to another.

This causes:

* Network I/O
* Extra disk I/O in some cases
* More processing
* More time

So, in simple terms:

> **Moving data inside the same worker is generally cheaper than moving data from one worker to another.**

This is why shuffle is considered an expensive operation in Spark.

---

# 4. Important idea

The original partitions are not necessarily kept during the shuffle.

Spark creates **new shuffle partitions** based on the shuffle operation.

For Spark SQL/DataFrame operations, the default number of shuffle partitions is commonly:

```text
200
```

This can be changed using:

```python
spark.conf.set("spark.sql.shuffle.partitions", 200)
```

The important point is not the number `200`.

The important point is:

```text
Same join key
      ↓
Same hash result
      ↓
Same shuffle partition
      ↓
Data can be joined
```

---

# 5. Final mental model

Think of shuffle like this:

```text
Data is scattered across different workers
                    ↓
             Join on `id`
                    ↓
        Spark checks the join key
                    ↓
      Data is redistributed using hash
                    ↓
 Same IDs are brought to the same partition
                    ↓
              Join happens
```

### One-line definition

> **Shuffle is the process of redistributing data across partitions (and sometimes across worker nodes) so that records with the same key are brought together for operations such as joins, groupBy, and aggregations.**


# Shuffle Sort Merge Join

Suppose we have two large DataFrames:

- DF1 = 500 MB
- DF2 = 500 MB
- DF1 has columns: `id`, `name`
- DF2 has columns: `id`, `salary`
- We are joining both DataFrames on `id`

When both DataFrames are large, Spark can use a **Shuffle Sort Merge Join**.

---

## 1. Shuffle

First, Spark needs to make sure that records with the same `id` are present in the same partition.

Suppose:

    Worker 1
    DF1 → id = 7

    Worker 2
    DF2 → id = 7

The matching records are on different workers.

Spark therefore performs a shuffle.

Conceptually:

    hash(id) % number_of_shuffle_partitions

For example:

    DF1 → id = 7 → Shuffle Partition 143
    DF2 → id = 7 → Shuffle Partition 143

Now both records with `id = 7` are in the same partition.

So:

    Same id
       ↓
    Same shuffle partition

The important point is:

> Shuffle brings matching keys into the same partition.

---

## 2. Sort

After the shuffle, both DataFrames have their matching keys in the same partitions.

However, the records inside those partitions may not be sorted.

For example:

    DF1:

    7
    2
    10
    5

    DF2:

    5
    10
    2
    7

Spark sorts both sides using the join key:

    DF1:

    2
    5
    7
    10

    DF2:

    2
    5
    7
    10

Now both sides are sorted by `id`.

The important point is:

> Sort puts the records in order so that Spark can efficiently compare the two sides.

---

## 3. Merge

Now Spark has two sorted datasets.

For example:

    DF1: 1  3  5  7  9

    DF2: 1  2  5  7  10

Spark goes through both sides and compares the values.

Conceptually:

    DF1 → 1
    DF2 → 1
    MATCH

    DF1 → 3
    DF2 → 2

    2 < 3
    Move forward in DF2

    DF2 → 5

    3 < 5
    Move forward in DF1

    DF1 → 5
    MATCH

    DF1 → 7
    DF2 → 7
    MATCH

Because both sides are sorted, Spark can scan through them efficiently instead of repeatedly searching the entire other dataset.

---

## 4. Complete Flow

The complete Shuffle Sort Merge Join looks like this:

    DF1                         DF2
     |                           |
     |                           |
     +--------- SHUFFLE ---------+
                |
                ↓
    Same join keys go to the
    same shuffle partitions
                |
                ↓
               SORT
                |
                ↓
    Both sides are sorted
       by the join key
                |
                ↓
              MERGE
                |
                ↓
             JOIN RESULT

---

## 5. What Does Each Step Actually Do?

### Shuffle

Purpose:

> Bring records with the same join key into the same partition.

Example:

    DF1 → id = 7 → Partition 143
    DF2 → id = 7 → Partition 143

### Sort

Purpose:

> Sort the records inside those partitions by the join key.

Example:

    Before:

    7
    2
    10
    5

    After:

    2
    5
    7
    10

### Merge

Purpose:

> Compare the two sorted datasets and find matching keys efficiently.

---

## 6. Important Point

The name **Shuffle Sort Merge Join** tells us exactly what is happening:

    SHUFFLE
       ↓
    Bring same keys into the same partitions

    SORT
       ↓
    Sort the data by the join key

    MERGE
       ↓
    Compare the sorted data and join matching records

So:

> Shuffle gets the right data together.

> Sort puts the data in order.

> Merge efficiently matches the data.

---

## 7. Does Sort-Merge Join Avoid Shuffle?

No.

A normal Shuffle Sort Merge Join can still require a shuffle.

The process is:

    Shuffle
       ↓
    Sort
       ↓
    Merge

So **Sort-Merge Join is not a method for avoiding shuffle**.

It is a method Spark uses to perform the join efficiently when dealing with large datasets.

---

## 8. Simple Example

Suppose:

    DF1:

    id = 1, name = John
    id = 7, name = Alex

    DF2:

    id = 1, salary = 50000
    id = 7, salary = 70000

After shuffle:

    Partition A:

    DF1 → id = 1
    DF2 → id = 1

    Partition B:

    DF1 → id = 7
    DF2 → id = 7

Then Spark sorts the partitions:

    Partition A:

    DF1 → id = 1
    DF2 → id = 1

    Partition B:

    DF1 → id = 7
    DF2 → id = 7

Then Spark merges the matching records:

    id = 1, John, 50000
    id = 7, Alex, 70000

---

## 9. Final Mental Model

Think of it as three separate jobs:

    SHUFFLE
    "Bring the same IDs together."

             ↓

    SORT
    "Put those IDs in order."

             ↓

    MERGE
    "Walk through both sorted sides and match the IDs."

The easiest way to remember it is:

> **Shuffle = bring together**

> **Sort = arrange**

> **Merge = match**

# Broadcast Hash Join

Suppose we have two DataFrames:

* `DF1` = 500 MB
* `DF2` = 10 MB
* `DF1` has columns: `id`, `name`
* `DF2` has columns: `id`, `salary`
* We are joining both DataFrames using `id`

Here, `DF2` is much smaller than `DF1`.

Instead of shuffling both DataFrames, Spark can use a **Broadcast Hash Join**.

---

## 1. The Problem with a Normal Join

Suppose the data is distributed like this:

```
Worker 1
├── DF1 Partition 1
└── DF1 Partition 2

Worker 2
├── DF1 Partition 3
└── DF1 Partition 4
```

And `DF2` is also distributed across workers.

Suppose we have a matching record:

```
DF1 → id = 7 → Worker 1
DF2 → id = 7 → Worker 2
```

In a normal shuffle join, Spark would need to move data so that both `id = 7` records end up in the same partition.

That means shuffle:

```
Worker 1                    Worker 2
DF1 → id = 7                DF2 → id = 7
     \                         /
      \                       /
       +------ SHUFFLE ------+
                |
                ↓
         Same Partition
                |
                ↓
               JOIN
```

The problem is that moving a large amount of data between workers is expensive.

---

## 2. The Basic Idea of Broadcast Join

Now suppose:

```
DF1 = 500 MB
DF2 = 10 MB
```

Instead of moving the large `DF1` around, Spark takes the small `DF2` and sends a copy of it to every worker.

This is called **broadcasting**.

```
DF2 = 10 MB
       |
       ↓
    BROADCAST
   /     |     \
  ↓      ↓      ↓
Worker 1 Worker 2 Worker 3
```

Now every worker has a copy of `DF2`.

The large `DF1` can stay distributed in its existing partitions.

---

## 3. Why Does This Help?

Without broadcast:

```
Large DF1
     +
Large DF2
     ↓
   SHUFFLE
     ↓
Move data between workers
     ↓
    JOIN
```

With broadcast:

```
Small DF2
     ↓
  BROADCAST
     ↓
Send DF2 to every worker
     ↓
DF1 stays where it is
     ↓
Local JOIN on each worker
```

So the main advantage is:

> **We avoid shuffling the large DataFrame.**

There is still network communication because the small DataFrame has to be sent to the workers, but sending 10 MB is much cheaper than shuffling hundreds of MBs of data.

---

## 4. What Does "Hash" Mean?

After the small DataFrame is broadcast to each worker, Spark creates a **hash table** from the small DataFrame.

Suppose `DF2` contains:

```
id | salary
-----------
1  | 50000
7  | 70000
10 | 90000
```

Spark can create something conceptually like:

```
Hash Table

1  → 50000
7  → 70000
10 → 90000
```

Here:

* `id` is the key
* `salary` is the value

The purpose of this hash table is to make finding a particular `id` very fast.

---

## 5. Joining With the Large DataFrame

Suppose Worker 1 has this partition of `DF1`:

```
id | name
-----------
3  | Mike
7  | Alex
15 | Sam
```

And Worker 1 already has the broadcasted hash table:

```
1  → 50000
7  → 70000
10 → 90000
```

Spark checks each `id` from the local `DF1` partition against the hash table.

For `id = 3`:

```
3 → Not found
```

For `id = 7`:

```
7 → Found
  → salary = 70000
```

For `id = 15`:

```
15 → Not found
```

So the result for this partition is:

```
id | name | salary
-------------------
7  | Alex | 70000
```

---

## 6. Complete Example

Suppose:

```
DF1:

id = 1, name = John
id = 7, name = Alex
id = 15, name = Sam

DF2:

id = 1, salary = 50000
id = 7, salary = 70000
```

`DF2` is small, so Spark broadcasts it.

Every worker gets:

```
Hash Table

1  → 50000
7  → 70000
```

Now suppose Worker 1 has:

```
DF1:

id = 1, name = John
id = 7, name = Alex
```

Spark performs hash lookups:

```
id = 1
  ↓
Found
  ↓
salary = 50000

id = 7
  ↓
Found
  ↓
salary = 70000
```

The result is:

```
id | name | salary
-------------------
1  | John | 50000
7  | Alex | 70000
```

No large shuffle of `DF1` was required.

---

## 7. Complete Flow

The whole Broadcast Hash Join process is:

```
Large DataFrame (DF1)
          |
          ↓
   Existing Partitions
          |
          |
          |
Small DataFrame (DF2)
          |
          ↓
     BROADCAST
          |
          ↓
Send DF2 to every worker
          |
          ↓
Build Hash Table from DF2
          |
          ↓
Each worker takes its local
partition of DF1
          |
          ↓
Look up each DF1.id
in the Hash Table
          |
          ↓
         JOIN
          |
          ↓
      Final Result
```

---

## 8. Why Is It Called Broadcast Hash Join?

The name describes the process:

```
BROADCAST
    ↓
Send the small DataFrame
to every worker

HASH
    ↓
Build a hash table using
the join key

JOIN
    ↓
Look up keys from the large
DataFrame in the hash table
```

So:

> **Broadcast → Hash → Join**

---

## 9. Broadcast Hash Join vs Shuffle Sort Merge Join

### Broadcast Hash Join

```
Small DataFrame
      ↓
  Broadcast
      ↓
Every worker gets a copy
      ↓
Build Hash Table
      ↓
Local Hash Lookup
      ↓
     JOIN
```

### Shuffle Sort Merge Join

```
DF1 + DF2
      ↓
   Shuffle
      ↓
Same keys go to the same
partitions
      ↓
    Sort
      ↓
    Merge
      ↓
     JOIN
```

The main difference is:

> **Broadcast Hash Join sends the small DataFrame to the workers instead of shuffling the large DataFrame.**

---

## 10. When Is Broadcast Hash Join Useful?

Broadcast Hash Join is useful when one DataFrame is much smaller than the other.

For example:

```
DF1 = 500 MB
DF2 = 10 MB
```

This is a good situation for broadcasting `DF2`.

But if both DataFrames are large:

```
DF1 = 500 MB
DF2 = 500 MB
```

Broadcasting either DataFrame may not be practical because every worker would need to keep a large copy in memory.

In that situation, Spark can use a Shuffle Sort Merge Join.

---

## 11. Using Broadcast Explicitly

We can explicitly tell Spark to broadcast a DataFrame:

```
from pyspark.sql.functions import broadcast

df1.join(broadcast(df2), "id")
```

Here:

```
df1 → Large DataFrame
df2 → Small DataFrame
```

Spark will broadcast `df2`.

Spark can also automatically choose a Broadcast Hash Join when the smaller DataFrame is below the configured broadcast threshold.

---

## 12. Important Point

Broadcast Hash Join does **not** mean that there is zero network communication.

The small DataFrame still needs to be sent to the workers.

The important difference is the **amount of data being moved**.

Instead of:

```
Large DataFrame
      ↓
   SHUFFLE
      ↓
Move lots of data
```

we do:

```
Small DataFrame
      ↓
  BROADCAST
      ↓
Send a small copy
to every worker
```

Therefore:

> **Broadcast Hash Join is useful because it avoids the expensive shuffle of the large DataFrame when one side of the join is small enough to broadcast.**

---

## 13. Final Mental Model

Think about it like this:

```
DF1 = BIG
DF2 = SMALL

        DF2
         |
         ↓
    BROADCAST
   /     |     \
  ↓      ↓      ↓
 W1     W2     W3
  |      |      |
  ↓      ↓      ↓
DF1    DF1    DF1
  |      |      |
  ↓      ↓      ↓
Hash   Hash   Hash
Lookup Lookup Lookup
  |      |      |
  ↓      ↓      ↓
Join   Join   Join
```

The easiest way to remember it:

> **Broadcast = send the small table to everyone.**

> **Hash = create a hash table from the small table.**

> **Join = look up the large table's keys in that hash table.**

> **Main benefit = avoid shuffling the large DataFrame.**
